#1: Imports and Path Setup

In [1]:

import os
import sys
import numpy as np
import tensorflow as tf


In [2]:
PROJECT_ROOT = '/content/drive/MyDrive/Transformer_Mine'
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints/small_scale')
TRANSLATIONS_DIR = os.path.join(PROJECT_ROOT, 'translations')

In [3]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Transformer_Mine

Mounted at /content/drive
/content/drive/MyDrive/Transformer_Mine


In [4]:
from transformer_model import TiedTransformer
from training import TransformerLRSchedule, create_checkpoint_manager, restore_checkpoint
from tokenizer import BPETokenizer
from data_pipeline import load_raw_pairs

os.makedirs(TRANSLATIONS_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Project root:", PROJECT_ROOT)

TensorFlow: 2.20.0
Project root: /content/drive/MyDrive/Transformer_Mine


#2: Load Model from Checkpoint

In [5]:
# ============================================================
# PHASE 14 - CELL 2: Load Model from H5 Weights (fixed)
# ============================================================

VOCAB_DIR_B = os.path.join(PROJECT_ROOT, 'vocab/stage_b')
tokenizer   = BPETokenizer.load(VOCAB_DIR_B)
VOCAB_SIZE  = tokenizer.vocab_size

MODEL_CONFIG = {
    'num_layers' : 3,
    'd_model'    : 256,
    'num_heads'  : 4,
    'd_ff'       : 1024,
    'dropout'    : 0.0,
    'max_len'    : 100,
}

model = TiedTransformer(
    num_layers = MODEL_CONFIG['num_layers'],
    d_model    = MODEL_CONFIG['d_model'],
    num_heads  = MODEL_CONFIG['num_heads'],
    d_ff       = MODEL_CONFIG['d_ff'],
    src_vocab  = VOCAB_SIZE,
    tgt_vocab  = VOCAB_SIZE,
    max_len    = MODEL_CONFIG['max_len'],
    dropout    = MODEL_CONFIG['dropout'],
)

# Build model with realistic token IDs before loading weights
safe_id   = min(100, VOCAB_SIZE - 1)
build_src = tf.constant([[safe_id] * 10], dtype=tf.int32)
build_tgt = tf.constant(
    [[tokenizer.bos_id] + [safe_id] * 9], dtype=tf.int32
)
_ = model(build_src, build_tgt, training=False)

# Load from h5 — reliable cross-session restore
WEIGHTS_PATH = os.path.join(
    PROJECT_ROOT,
    'checkpoints/small_scale/weights_final.weights.h5'
)
model.load_weights(WEIGHTS_PATH)
print(f"Weights loaded from: {WEIGHTS_PATH}")
print(f"  File size : {os.path.getsize(WEIGHTS_PATH)/1_048_576:.2f} MB")

# Verify predictions look reasonable
test_ids    = tokenizer.encode_with_special("I am a student .")
test_tensor = tf.constant(
    [test_ids + [0] * (100 - len(test_ids))], dtype=tf.int32
)
enc_mask_v  = tf.cast(
    tf.equal(test_tensor, 0), tf.float32
)[:, tf.newaxis, tf.newaxis, :]
enc_out_v, _ = model.encoder(test_tensor, mask=enc_mask_v, training=False)
dec_v        = tf.constant([[tokenizer.bos_id]], dtype=tf.int32)
dec_mask_v   = 1.0 - tf.linalg.band_part(tf.ones((1,1,1,1)), -1, 0)
dec_out_v, _ = model.decoder(
    dec_v, enc_out_v,
    dec_mask=dec_mask_v, enc_mask=enc_mask_v, training=False
)
logits_v = tf.matmul(
    dec_out_v[:, -1:, :],
    model.decoder.embedding.embeddings,
    transpose_b=True
).numpy()[0, 0]

top5 = np.argsort(logits_v)[-5:][::-1]
print(f"\nTop 5 first token predictions for 'I am a student .':")
for tok in top5:
    print(f"  '{tokenizer.id2token.get(int(tok), '?')}'  logit={logits_v[tok]:.4f}")

total_params = sum(tf.size(w).numpy() for w in model.trainable_variables)
print(f"\nParameters: {total_params:,}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'decoder_cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will no

Weights loaded from: /content/drive/MyDrive/Transformer_Mine/checkpoints/small_scale/weights_final.weights.h5
  File size : 36.94 MB

Top 5 first token predictions for 'I am a student .':
  'Ich</w>'  logit=11.2923
  'Tom</w>'  logit=3.3370
  'Man</w>'  logit=2.5476
  'Meine</w>'  logit=2.3472
  'Maria</w>'  logit=2.3418

Parameters: 9,616,384


# 3: Encoder Helper

Runs the encoder once per source sentence.

Both greedy and beam search share this same encoder output.


In [6]:
def encode_source(model, src_text, tokenizer, max_len=100):
    """
    Tokenise and encode a source sentence through the encoder.

    Args:
        model : TiedTransformer
        src_text : raw source string
        tokenizer : BPETokenizer
        max_len : max source length

    Returns:
        enc_output : (1, src_len, d_model)
        enc_mask : (1, 1, 1, src_len)
        src_ids : (1, src_len) integer tensor
    """
    # Tokenise
    src_ids = tokenizer.encode_with_special(src_text)
    src_ids = src_ids[:max_len]

    # Pad to max_len
    src_ids = src_ids + [tokenizer.pad_id] * (max_len - len(src_ids))
    src_tensor = tf.constant([src_ids], dtype=tf.int32)

    # Padding mask
    enc_mask = tf.cast(
        tf.equal(src_tensor, tokenizer.pad_id), tf.float32
    )[:, tf.newaxis, tf.newaxis, :]

    # Encoder forward pass
    enc_output, _ = model.encoder(src_tensor, mask=enc_mask, training=False)

    return enc_output, enc_mask, src_tensor

In [7]:


def decoder_mask_for(dec_ids, pad_id):
    """Build combined causal + padding mask for current decoder ids."""
    tgt_len = tf.shape(dec_ids)[1]
    look_ahead = 1.0 - tf.linalg.band_part(
        tf.ones((1, 1, tgt_len, tgt_len)), -1, 0
    )
    pad_mask = tf.cast(
        tf.equal(dec_ids, pad_id), tf.float32
    )[:, tf.newaxis, tf.newaxis, :]

    return tf.maximum(look_ahead, pad_mask)

In [8]:
def project_to_vocab(model, dec_output_last):
    """
    Project decoder last position output to vocabulary logits.
    Uses tied embedding weights (paper Section 3.4).

    Args:
        dec_output_last : (batch, 1, d_model)

    Returns:
        logits : (batch, vocab_size)
    """
    embedding_weights = model.decoder.embedding.embeddings
    logits = tf.matmul(dec_output_last, embedding_weights, transpose_b=True)

    return logits[:, 0, :]   # (batch, vocab_size)


# 4: Greedy Decoding with Repeat Penalty

Greedy decoding: at each step, pick the highest-probability token. Fast but suboptimal — tends to produce repetition loops for rare or long target words.

Repeat penalty: subtract a fixed value from logits of recently generated tokens before taking argmax. Prevents loops.


In [9]:


def greedy_decode(
    model,
    src_text,
    tokenizer,
    max_len = 50,
    repeat_penalty = 2.0,
    repeat_window = 5,
):
    """
    Greedy decoding with repeat penalty.

    Args:
        model : TiedTransformer
        src_text : raw source string
        tokenizer : BPETokenizer
        max_len : maximum output tokens
        repeat_penalty : logit penalty applied to recent tokens
        repeat_window : how many recent tokens to penalise

    Returns:
        decoded string
    """
    enc_output, enc_mask, _ = encode_source(model, src_text, tokenizer)

    dec_ids = tf.constant([[tokenizer.bos_id]], dtype=tf.int32)
    generated = []

    for _ in range(max_len):
        dec_mask = decoder_mask_for(dec_ids, tokenizer.pad_id)
        dec_output, _ = model.decoder(
            dec_ids, enc_output,
            dec_mask=dec_mask,
            enc_mask=enc_mask,
            training=False,
        )

        logits = project_to_vocab(model, dec_output[:, -1:, :]).numpy()[0]

        # Apply repeat penalty to recent tokens
        for tok in set(generated[-repeat_window:]):
            logits[tok] -= repeat_penalty

        next_token = int(np.argmax(logits))

        if next_token == tokenizer.eos_id:
            break

        generated.append(next_token)
        dec_ids = tf.concat(
            [dec_ids, tf.constant([[next_token]], dtype=tf.int32)],
            axis=1,
        )

    return tokenizer.decode(generated)



# 5: Beam Search

Beam search maintains the top-k partial hypotheses at each step, expanding each by one token and keeping the k best.

This finds a better sequence than greedy because it does not commit to the single highest-probability token at each step.

From the paper (Section 5.4):

* Beam size = 4
* Length penalty alpha = 0.6

**Length penalty:**

```text
score = log_prob / length_penalty(len)
length_penalty(len) = ((5 + len) / (5 + 1))^alpha
```

Without length penalty, beam search prefers short sequences because longer sequences accumulate more log probability mass.


In [10]:

def length_penalty(length, alpha=0.6):
    """
    Length penalty from paper Section 5.4.
    ((5 + length) / (5 + 1))^alpha
    """

    return ((5.0 + length) / 6.0) ** alpha


def beam_search(
    model,
    src_text,
    tokenizer,
    beam_size = 4,
    max_len = 50,
    alpha = 0.6,
    min_len = 1,
):
    """
    Beam search decoding.

    Args:
        model : TiedTransformer
        src_text : raw source string
        tokenizer : BPETokenizer
        beam_size : number of beams (paper: 4)
        max_len : maximum decode length
        alpha : length penalty exponent (paper: 0.6)
        min_len : minimum decode length before EOS is allowed

    Returns:
        best decoded string
    """
    enc_output, enc_mask, _ = encode_source(model, src_text, tokenizer)

    # Expand encoder output for beam_size hypotheses
    # enc_output: (1, src_len, d_model) -> (beam_size, src_len, d_model)
    enc_output = tf.repeat(enc_output, beam_size, axis=0)
    enc_mask = tf.repeat(enc_mask,   beam_size, axis=0)

    # Each beam: (token_ids, cumulative_log_prob)
    beams = [([tokenizer.bos_id], 0.0)]
    completed = []

    for step in range(max_len):
        all_candidates = []

        for beam_ids, beam_score in beams:
            # Build decoder input tensor for this beam
            dec_ids = tf.constant([beam_ids], dtype=tf.int32)
            dec_mask = decoder_mask_for(dec_ids, tokenizer.pad_id)

            dec_output, _ = model.decoder(
                dec_ids, enc_output[:1],   # use single encoder output
                dec_mask=dec_mask,
                enc_mask=enc_mask[:1],
                training=False,
            )

            logits    = project_to_vocab(
                model, dec_output[:, -1:, :]
            ).numpy()[0]
            log_probs = logits - np.log(np.sum(np.exp(logits - logits.max())) ) - logits.max()
            # numerically stable log_softmax
            log_probs = logits - (
                np.log(np.sum(np.exp(logits - logits.max()))) + logits.max()
            )

            # Top beam_size tokens for this beam
            top_tokens = np.argsort(log_probs)[-beam_size:]

            for token in top_tokens:
                new_ids   = beam_ids + [int(token)]
                new_score = beam_score + float(log_probs[token])

                if int(token) == tokenizer.eos_id and len(new_ids) > min_len:
                    # Completed hypothesis — apply length penalty
                    penalty   = length_penalty(len(new_ids), alpha)
                    final_score = new_score / penalty
                    completed.append((new_ids, final_score))
                else:
                    all_candidates.append((new_ids, new_score))

        if not all_candidates:
            break

        # Keep top beam_size candidates
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_size]

        # Early stop if we have enough completed hypotheses
        if len(completed) >= beam_size:
            break

    # If no hypothesis completed, use best active beam
    if not completed:
        best_ids = beams[0][0]
    else:
        completed.sort(key=lambda x: x[1], reverse=True)
        best_ids = completed[0][0]

    # Remove BOS and EOS
    best_ids = [t for t in best_ids
                if t not in (tokenizer.bos_id, tokenizer.eos_id)]

    return tokenizer.decode(best_ids)



#6: Compare Greedy vs Beam Search

In [11]:
STAGE_B_DIR = os.path.join(PROJECT_ROOT, 'datasets/stage_b')
src_val, tgt_val = load_raw_pairs(
    os.path.join(STAGE_B_DIR, 'val.en'),
    os.path.join(STAGE_B_DIR, 'val.de'),
)

In [12]:

test_sentences = src_val[:15]
gold_targets = tgt_val[:15]



In [13]:
results = []

In [14]:
for src, tgt in zip(test_sentences, gold_targets):
    greedy_out = greedy_decode(model, src, tokenizer, max_len=50)
    beam_out = beam_search(model, src, tokenizer,
                              beam_size=4, max_len=50, alpha=0.6)

    results.append({
        'src' : src,
        'tgt' : tgt,
        'greedy': greedy_out,
        'beam' : beam_out,
    })

    print(f"src    : {src}")
    print(f"tgt    : {tgt}")
    print(f"greedy : {greedy_out}")
    print(f"beam   : {beam_out}")

src    : She left.
tgt    : Sie ist gegangen.
greedy : Sie ist gegangen .
beam   : Sie ist gegangen .
src    : Where are you all?
tgt    : Wo seid ihr alle?
greedy : Wo sind Sie ?
beam   : Wo sind Sie ?
src    : Can we believe that?
tgt    : Können wir das glauben?
greedy : Können wir das glauben ?
beam   : Können wir das glauben ?
src    : Tom stood rigid.
tgt    : Tom stand unbeweglich.
greedy : Tom hat bessercht .
beam   : Tom hat bessercht .
src    : He likes buying notebooks.
tgt    : Er kauft gerne Notizblöcke.
greedy : Er mag gerne Notizbücher .
beam   : Er mag gerne Notizbücher .
src    : I'm being honest.
tgt    : Ich bin ehrlich.
greedy : Ich bin ehrlich .
beam   : Ich bin ehrlich .
src    : Do you like this?
tgt    : Magst du das?
greedy : Magst du das ?
beam   : Magst du das ?
src    : Someone's talking.
tgt    : Jemand redet.
greedy : Jemand spricht .
beam   : Jemand spricht .
src    : Tom stayed seated.
tgt    : Tom blieb sitzen.
greedy : Tom blieb sitzen .
beam   : Tom b

#  7: Effect of Beam Size

Tests beam sizes 1, 2, 4, and 8 on a handful of examples.

`beam_size=1` is equivalent to greedy decoding.


In [15]:
test_src = src_val[:5]
test_tgt = tgt_val[:5]

In [16]:

for src, tgt in zip(test_src, test_tgt):
    print(f"src : {src}")
    print(f"tgt : {tgt}")
    for beam_size in [1, 2, 4, 8]:
        out = beam_search(model, src, tokenizer,
                          beam_size=beam_size, max_len=50)
        print(f"  beam={beam_size} : {out}")

src : She left.
tgt : Sie ist gegangen.
  beam=1 : Sie ist gegangen .
  beam=2 : Sie ist gegangen .
  beam=4 : Sie ist gegangen .
  beam=8 : Sie ist gegangen .
src : Where are you all?
tgt : Wo seid ihr alle?
  beam=1 : Wo sind Sie ?
  beam=2 : Wo sind Sie ?
  beam=4 : Wo sind Sie ?
  beam=8 : Wo sind Sie ?
src : Can we believe that?
tgt : Können wir das glauben?
  beam=1 : Können wir das glauben ?
  beam=2 : Können wir das glauben ?
  beam=4 : Können wir das glauben ?
  beam=8 : Können wir das glauben ?
src : Tom stood rigid.
tgt : Tom stand unbeweglich.
  beam=1 : Tom hat bessercht .
  beam=2 : Tom hat bessercht .
  beam=4 : Tom hat bessercht .
  beam=8 : Tom hat bessercht .
src : He likes buying notebooks.
tgt : Er kauft gerne Notizblöcke.
  beam=1 : Er mag gerne Notizbücher .
  beam=2 : Er mag gerne Notizbücher .
  beam=4 : Er mag gerne Notizbücher .
  beam=8 : Er mag gerne Notibücher .


# 8: Effect of Length Penalty

Tests alpha values 0.0, 0.3, 0.6, and 0.9 on a few examples.

* `alpha=0.0` means no length penalty (prefers shorter output).
* `alpha=1.0` means full length normalisation.
* Paper uses `alpha=0.6`.


In [17]:
test_src = src_val[:3]
test_tgt = tgt_val[:3]

In [18]:
for src, tgt in zip(test_src, test_tgt):
    print(f"src : {src}")
    print(f"tgt : {tgt}")
    for alpha in [0.0, 0.3, 0.6, 0.9]:
        out = beam_search(model, src, tokenizer,
                          beam_size=4, max_len=50, alpha=alpha)
        print(f"  alpha={alpha} : {out}")
    print()

src : She left.
tgt : Sie ist gegangen.
  alpha=0.0 : Sie ist gegangen .
  alpha=0.3 : Sie ist gegangen .
  alpha=0.6 : Sie ist gegangen .
  alpha=0.9 : Sie ist gegangen .

src : Where are you all?
tgt : Wo seid ihr alle?
  alpha=0.0 : Wo sind Sie ?
  alpha=0.3 : Wo sind Sie ?
  alpha=0.6 : Wo sind Sie ?
  alpha=0.9 : Wo sind Sie ?

src : Can we believe that?
tgt : Können wir das glauben?
  alpha=0.0 : Können wir das glauben ?
  alpha=0.3 : Können wir das glauben ?
  alpha=0.6 : Können wir das glauben ?
  alpha=0.9 : Können wir das glauben ?



#9: Save Translations to Drive

In [22]:

import json
import time

print("Generating translations for validation set...")
print(f"  {len(src_val[:500])} sentences")
print()

greedy_translations = []
beam_translations   = []

t0 = time.time()

for i, src in enumerate(src_val[:500]):
    greedy_translations.append(
        greedy_decode(model, src, tokenizer, max_len=50)
    )
    beam_translations.append(
        beam_search(model, src, tokenizer,
                    beam_size=4, max_len=50, alpha=0.6)
    )

    elapsed = time.time() - t0
    print(f"  {i+1}/{len(src_val[:500])}  elapsed={elapsed:.0f}s")

elapsed = time.time() - t0
print(f"\nDone. {len(src_val[:500])} translations in {elapsed:.0f}s")

greedy_path = os.path.join(TRANSLATIONS_DIR, 'val_greedy.txt')
beam_path   = os.path.join(TRANSLATIONS_DIR, 'val_beam4.txt')
ref_path    = os.path.join(TRANSLATIONS_DIR, 'val_reference.txt')

with open(greedy_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(greedy_translations))

with open(beam_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(beam_translations))

with open(ref_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(tgt_val))

print(f"\nSaved:")
print(f"  greedy translations : {greedy_path}")
print(f"  beam translations   : {beam_path}")
print(f"  reference targets   : {ref_path}")

Generating translations for validation set...
  500 sentences

  1/500  elapsed=3s
  2/500  elapsed=6s
  3/500  elapsed=8s
  4/500  elapsed=12s
  5/500  elapsed=17s
  6/500  elapsed=19s
  7/500  elapsed=22s
  8/500  elapsed=24s
  9/500  elapsed=26s
  10/500  elapsed=29s
  11/500  elapsed=32s
  12/500  elapsed=34s
  13/500  elapsed=38s
  14/500  elapsed=41s
  15/500  elapsed=44s
  16/500  elapsed=48s
  17/500  elapsed=51s
  18/500  elapsed=54s
  19/500  elapsed=56s
  20/500  elapsed=59s
  21/500  elapsed=61s
  22/500  elapsed=64s
  23/500  elapsed=66s
  24/500  elapsed=70s
  25/500  elapsed=75s
  26/500  elapsed=78s
  27/500  elapsed=81s
  28/500  elapsed=84s
  29/500  elapsed=87s
  30/500  elapsed=89s
  31/500  elapsed=92s
  32/500  elapsed=94s
  33/500  elapsed=97s
  34/500  elapsed=102s
  35/500  elapsed=106s
  36/500  elapsed=109s
  37/500  elapsed=111s
  38/500  elapsed=114s
  39/500  elapsed=118s
  40/500  elapsed=121s
  41/500  elapsed=125s
  42/500  elapsed=128s
  43/500  elapse

# 10: Save Decoding Utilities to Drive

In [23]:
DECODING_PY = os.path.join(PROJECT_ROOT, 'decoding.py')

In [24]:



code = '''"""
decoding.py - Greedy and Beam Search Decoding
Transformer Reproduction Project - Phase 14

Usage:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/Transformer_Mine
    from decoding import greedy_decode, beam_search
"""

import numpy as np
import tensorflow as tf


def encode_source(model, src_text, tokenizer, max_len=100):
    src_ids    = tokenizer.encode_with_special(src_text)[:max_len]
    src_ids    = src_ids + [tokenizer.pad_id] * (max_len - len(src_ids))
    src_tensor = tf.constant([src_ids], dtype=tf.int32)
    enc_mask   = tf.cast(
        tf.equal(src_tensor, tokenizer.pad_id), tf.float32
    )[:, tf.newaxis, tf.newaxis, :]
    enc_output, _ = model.encoder(src_tensor, mask=enc_mask, training=False)
    return enc_output, enc_mask, src_tensor


def decoder_mask_for(dec_ids, pad_id):
    tgt_len    = tf.shape(dec_ids)[1]
    look_ahead = 1.0 - tf.linalg.band_part(
        tf.ones((1, 1, tgt_len, tgt_len)), -1, 0
    )
    pad_mask = tf.cast(
        tf.equal(dec_ids, pad_id), tf.float32
    )[:, tf.newaxis, tf.newaxis, :]
    return tf.maximum(look_ahead, pad_mask)


def project_to_vocab(model, dec_output_last):
    weights = model.decoder.embedding.embeddings
    logits  = tf.matmul(dec_output_last, weights, transpose_b=True)
    return logits[:, 0, :]


def length_penalty(length, alpha=0.6):
    return ((5.0 + length) / 6.0) ** alpha


def greedy_decode(model, src_text, tokenizer,
                  max_len=50, repeat_penalty=2.0, repeat_window=5):
    enc_output, enc_mask, _ = encode_source(model, src_text, tokenizer)
    dec_ids   = tf.constant([[tokenizer.bos_id]], dtype=tf.int32)
    generated = []

    for _ in range(max_len):
        dec_mask   = decoder_mask_for(dec_ids, tokenizer.pad_id)
        dec_output, _ = model.decoder(
            dec_ids, enc_output,
            dec_mask=dec_mask, enc_mask=enc_mask, training=False,
        )
        logits = project_to_vocab(model, dec_output[:, -1:, :]).numpy()[0]
        for tok in set(generated[-repeat_window:]):
            logits[tok] -= repeat_penalty
        next_token = int(np.argmax(logits))
        if next_token == tokenizer.eos_id:
            break
        generated.append(next_token)
        dec_ids = tf.concat(
            [dec_ids, tf.constant([[next_token]], dtype=tf.int32)], axis=1
        )
    return tokenizer.decode(generated)


def beam_search(model, src_text, tokenizer,
                beam_size=4, max_len=50, alpha=0.6, min_len=1):
    enc_output, enc_mask, _ = encode_source(model, src_text, tokenizer)
    beams     = [([tokenizer.bos_id], 0.0)]
    completed = []

    for step in range(max_len):
        all_candidates = []
        for beam_ids, beam_score in beams:
            dec_ids  = tf.constant([beam_ids], dtype=tf.int32)
            dec_mask = decoder_mask_for(dec_ids, tokenizer.pad_id)
            dec_output, _ = model.decoder(
                dec_ids, enc_output,
                dec_mask=dec_mask, enc_mask=enc_mask, training=False,
            )
            logits = project_to_vocab(model, dec_output[:, -1:, :]).numpy()[0]
            log_probs = logits - (
                np.log(np.sum(np.exp(logits - logits.max()))) + logits.max()
            )
            for token in np.argsort(log_probs)[-beam_size:]:
                new_ids   = beam_ids + [int(token)]
                new_score = beam_score + float(log_probs[token])
                if int(token) == tokenizer.eos_id and len(new_ids) > min_len:
                    completed.append((new_ids, new_score / length_penalty(len(new_ids), alpha)))
                else:
                    all_candidates.append((new_ids, new_score))

        if not all_candidates:
            break
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_size]
        if len(completed) >= beam_size:
            break

    if not completed:
        best_ids = beams[0][0]
    else:
        completed.sort(key=lambda x: x[1], reverse=True)
        best_ids = completed[0][0]

    best_ids = [t for t in best_ids
                if t not in (tokenizer.bos_id, tokenizer.eos_id)]
    return tokenizer.decode(best_ids)
'''

with open(DECODING_PY, 'w', encoding='utf-8') as f:
    f.write(code)

print(f"decoding.py saved to: {DECODING_PY}")
print()

decoding.py saved to: /content/drive/MyDrive/Transformer_Mine/decoding.py

